In [1]:
import pandas as pd

customers = pd.read_csv("../data/customers_clean.csv")

choose X and y

In [2]:
features = ["auto_renew", "avg_discount", "plan_days", "age", "city",
            "registered_via", "tenure_days", "total_secs", "active_days", "unique_songs"]

X = customers[features]
y = customers["is_churn"]

In [3]:
print(X.shape)
print(y.value_counts())

(970960, 10)
is_churn
0    883630
1     87330
Name: count, dtype: int64


split the data into learning and testing

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

the lazy baseline/model

In [6]:
from sklearn.dummy import DummyClassifier

lazy = DummyClassifier(strategy="most_frequent")
lazy.fit(X_train, y_train)
lazy_pred = lazy.predict(X_test)

In [7]:
from sklearn.metrics import accuracy_score, recall_score

print("accuracy:", round(accuracy_score(y_test, lazy_pred) * 100, 2), "%")
print("leavers caught:", round(recall_score(y_test, lazy_pred) * 100, 2), "%")

accuracy: 91.01 %
leavers caught: 0.0 %


train the real model (LightGBM)

In [8]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1)
model.fit(X_train, y_train)

,class_weight,'balanced'
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,min_split_gain,0.0


get a risk score for each test customer

In [9]:
risk = model.predict_proba(X_test)[:, 1]
print(risk[:10])

[0.27407903 0.19650544 0.13439626 0.35752594 0.39762295 0.34965507
 0.17733104 0.1467416  0.37945948 0.49856561]


how good is the model?

In [10]:
from sklearn.metrics import roc_auc_score, precision_score

pred = (risk >= 0.5).astype(int)

print("AUC:", round(roc_auc_score(y_test, risk), 3))
print("leavers caught (recall):", round(recall_score(y_test, pred) * 100, 1), "%")
print("correct when it says 'leaving' (precision):", round(precision_score(y_test, pred) * 100, 1), "%")

AUC: 0.877
leavers caught (recall): 72.7 %
correct when it says 'leaving' (precision): 36.4 %


In [11]:
importance = pd.Series(model.feature_importances_, index=features)
importance.sort_values(ascending=False)

tenure_days       704
total_secs        359
active_days       348
age               326
plan_days         296
unique_songs      292
registered_via    282
avg_discount      161
city              128
auto_renew        104
dtype: int32

In [12]:
gain = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=features)
(gain / gain.sum() * 100).round(1).sort_values(ascending=False)

plan_days         77.0
tenure_days        6.3
auto_renew         4.6
active_days        3.6
registered_via     2.8
avg_discount       2.3
city               1.2
unique_songs       0.9
age                0.8
total_secs         0.6
dtype: float64

In [13]:
missing = customers["plan_days"].isna()
print("customers with no payment info:", missing.sum())
customers.groupby(missing)["is_churn"].mean() * 100

customers with no payment info: 37382


plan_days
False     6.201410
True     78.741105
Name: is_churn, dtype: float64

## First model and a leakage check

- My first LightGBM model scored AUC 0.877, caught 72.7% of leavers, and was right 36.4% of the time when it flagged someone (vs 9% at random).
- But one input, plan_days, carried 77% of the model's importance, which looked suspicious.
- I checked: 37,382 customers had no payment in the file, and 78.7% of them churned, compared with 6.2% of customers who had a payment.
- A missing renewal payment is basically the outcome itself, so the payment inputs leak future information and the 0.877 is too optimistic.
- Next, I will rebuild the payment inputs using only payments made before each customer's renewal date, and retrain.

In [14]:
t = pd.read_csv("../data/transactions_v2.csv")
t["month"] = t["transaction_date"] // 100
t["month"].value_counts().sort_index().tail(12)

month
201604       7967
201605       7431
201606       7140
201607       8516
201608       8134
201609      10586
201610      17743
201611      18819
201612      23787
201701      32871
201702     128916
201703    1069822
Name: count, dtype: int64

In [ ]:
before = t[t["transaction_date"] < 20170301]
print("customers with a pre-March payment:", before["msno"].nunique())
print("% of our customers covered:", round(customers["msno"].isin(before["msno"]).mean() * 100, 1))


customers with a pre-March payment: 262429
% of our customers covered: 9.5


I rebuilt the payment features in Databricks (02b_payment_history) using only payments before 1 March 2017. Below, I retrain the model on this leak-free data

In [18]:
customers = pd.read_csv("../data/customers_clean_v2.csv")

In [19]:
features = ["plan_days", "auto_renew", "avg_discount", "ever_cancelled", "num_payments", "total_paid",
            "age", "city", "registered_via", "tenure_days", "total_secs", "active_days", "unique_songs"]

X = customers[features]
y = customers["is_churn"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1)
model.fit(X_train, y_train)

risk = model.predict_proba(X_test)[:, 1]
pred = (risk >= 0.5).astype(int)

print("AUC:", round(roc_auc_score(y_test, risk), 3))
print("recall:", round(recall_score(y_test, pred) * 100, 1), "%")
print("precision:", round(precision_score(y_test, pred) * 100, 1), "%")

AUC: 0.891
recall: 77.5 %
precision: 33.2 %


In [21]:
gain = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=features)
(gain / gain.sum() * 100).round(1).sort_values(ascending=False)

auto_renew        49.1
active_days        8.3
ever_cancelled     7.6
plan_days          7.1
num_payments       6.4
total_paid         5.5
avg_discount       4.3
tenure_days        3.8
registered_via     2.3
unique_songs       2.1
total_secs         1.8
city               1.2
age                0.6
dtype: float64

After removing leakage, the model relies mainly on pre-March auto-renew behaviour (49%), followed by listening activity and payment history.

In [22]:
no_listen = [f for f in features if f not in ["active_days", "total_secs", "unique_songs"]]

model2 = LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1)
model2.fit(X_train[no_listen], y_train)

risk2 = model2.predict_proba(X_test[no_listen])[:, 1]
print("AUC without listening:", round(roc_auc_score(y_test, risk2), 3))

AUC without listening: 0.871


Without the March listening columns, AUC is 0.871 (vs 0.891 with them). I use the 0.871 model as the final one, because it uses no information from the prediction month. Auto-renew behaviour before March is the strongest driver

## Explaining the model with SHAP

My model gives every customer a churn risk score, but a score alone isn't enough. The business needs to know why a customer is at risk before deciding what to do about it.

I used SHAP values to break each prediction into the contribution of each input. A positive value means that input pushed the customer's risk up, and a negative value means it pulled the risk down. LightGBM can calculate these directly with pred_contrib=True, so I didn't need an extra library.

I calculated SHAP values for 2,000 random test customers to see which inputs matter most overall, and then looked at one customer in detail to read their personal reasons.

calculating
 SHAP values for 2,000 customers

In [23]:
X_sample = X_test[no_listen].sample(2000, random_state=42)

contrib = model2.predict(X_sample, pred_contrib=True)
shap_df = pd.DataFrame(contrib[:, :-1], columns=no_listen)

In [24]:
shap_df.abs().mean().sort_values(ascending=False)

auto_renew        0.558003
ever_cancelled    0.342371
total_paid        0.235495
plan_days         0.194748
num_payments      0.184054
tenure_days       0.135703
registered_via    0.089854
avg_discount      0.079380
city              0.068525
age               0.045365
dtype: float64

In [25]:
print(X_sample.iloc[0])
shap_df.iloc[0].sort_values(key=abs, ascending=False).head(5)

plan_days           28.578947
auto_renew           1.000000
avg_discount        -7.842105
ever_cancelled       0.000000
num_payments        19.000000
total_paid        2831.000000
age                 28.000000
city                13.000000
registered_via       9.000000
tenure_days       4514.000000
Name: 903394, dtype: float64


auto_renew       -0.347920
ever_cancelled   -0.271241
num_payments     -0.130344
total_paid        0.096017
avg_discount     -0.069172
Name: 0, dtype: float64

This customer is low risk mainly because auto-renew has always been on and they’ve never cancelled, backed by 19 steady payments. Their high total spend nudges the risk up slightly, but not enough to matter

a risk score for every customer

In [27]:
customers["risk"] = model2.predict_proba(customers[no_listen])[:, 1]

In [28]:
import numpy as np

customers["monthly_value"] = (customers["total_paid"] / customers["num_payments"]) * (30 / customers["plan_days"])
customers["monthly_value"] = customers["monthly_value"].replace([np.inf, -np.inf], 0).fillna(0)

In [29]:
customers["value_at_risk"] = customers["monthly_value"] * 12 * customers["risk"]

In [30]:
print("total yearly revenue at risk (NT$):", round(customers["value_at_risk"].sum()))
customers.sort_values("value_at_risk", ascending=False)[["monthly_value", "risk", "value_at_risk"]].head(10)

total yearly revenue at risk (NT$): 534396174


,monthly_value,risk,value_at_risk
831808,1968.000000,0.926568,21881.825739
276684,1968.000000,0.908031,21444.053485
537132,1968.000000,0.907596,21433.791209
941508,1937.000000,0.775638,18028.918845
951849,1968.000000,0.755356,17838.495076
272933,1595.675676,0.904042,17310.688100
819603,1728.000000,0.786328,16305.290324
156372,1748.000000,0.753522,15805.868525
405009,1595.675676,0.797354,15267.819538
205443,1074.000000,0.893476,11515.122007


## Customer value

I estimated each customer's monthly value from their pre-March payments and combined it with their churn risk to get value at risk (12 months of payments × risk). In total, about NT$534 million a year is at risk. This is an upper estimate, since my risk scores run slightly high. The top customers pay around NT$1,600–2,000 a month with 75–93% risk, so they should be first in line for retention offers. I still need to check whether those very high monthly values are genuine plans.